# Exit-policy study, stage R: move vs implied range (I) and the rejection-wick exit (J)

**Question (roadmap step 4, rows I and J of the exhaustion brainstorm):** inside a trade, does the first minute at
which the favourable move exceeds one option-implied daily move, or the first 15-minute bar that rejects from a
causal level while the trade is ≥ 0.3 R in profit, say anything about what the rest of the trade is worth?

This notebook is a check on the frozen run, not a second run. It reloads `results/range_wick/events.csv.gz` and
recomputes each decision line's mean, interval and halves with the same block bootstrap, then asserts they match
`report.json`. Design: [PREREGISTRATION_RANGE_WICK.md](PREREGISTRATION_RANGE_WICK.md) v1.0 with amendment A1.
Verdict and reading: [findings_range_wick.md](findings_range_wick.md).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); sys.path.insert(0, str(HERE))
import micro_lib as M, rangewick_lib as L
R = HERE / "results" / "range_wick"
report = json.loads((R / "report.json").read_text())
verdict = json.loads((R / "verdict.json").read_text())
f0 = json.loads((R / "freeze_F0.json").read_text())
pre = json.loads((R / "preconditions.json").read_text())
events = pd.read_csv(R / "events.csv.gz")
trades = pd.read_csv(R / "trades.csv.gz")
print("frozen", f0["created_utc"], "| outcome", report["created_utc"])
print("family:", report["family"])
print("decision rungs:", f0["decision_rungs"])
print("verdict:", verdict["verdict"])
print("labels:", json.dumps(verdict["classifications"], indent=1))

frozen 2026-09-18T23:25:14+00:00 | outcome 2026-09-18T23:27:14+00:00
family: ['chento_BTC:I1_implied', 'chento_BTC:J_wick', 'squeeze_bull:J_wick']
decision rungs: {'chento_BTC:I1_implied': 'rung3', 'chento_BTC:J_wick': 'rung1', 'squeeze_bull:I1_implied': None, 'squeeze_bull:J_wick': 'rung1'}
verdict: NONE PROMOTED
labels: {
 "chento_BTC:I1_implied": "UNDETERMINED",
 "chento_BTC:J_wick": "UNDETERMINED",
 "squeeze_bull:J_wick": "UNDETERMINED"
}


## 1. The preconditions and the family

Eight checks, all passing after amendment A1 (the squeeze_bull population is the corrected-open-interest re-cut;
the imported libraries are recorded as they stand). The family was fixed on counts alone: three tests reached 30
included event trades under a decision rung; squeeze_bull's implied-range test did not (15 events, 7 of them at
the target minute) and is descriptive.

In [2]:
print("preconditions:", {k: pre[k]["pass"] for k in pre if k.startswith("P")})
print("populations:", pre["P2"]["counts"], "| I-eligible:", pre["P2"]["i_eligible"])
print("squeeze_bull vs stage 1:", {k: v for k, v in pre["P2"]["squeeze_bull_vs_stage1"].items()})
print("walker identity:", pre["P3"]["differences"], "compared", pre["P3"]["compared"])
print("DVOL:", {a: (v["first_day"], v["last_day"], v["rows"], v["missing_days_inside_span"]) for a, v in pre["P4"]["facts"].items()})
rows = []
for name, per in pre["P8"]["per_subpop"].items():
    for kind in ("I1_implied", "I1_realised", "I2_day", "J_wick", "J_nolevel", "J_accept", "J_shape", "J_wick_1m"):
        e = per[kind]; inc = e["included"]
        rows.append({"subpop": name, "kind": kind, "eligible": e["eligible_trades"], "events": e["trades_with_event"],
                     "median h": e.get("median_elapsed_hours"), "rung1": inc.get("rung1"), "rung2": inc.get("rung2"),
                     "rung3": inc.get("rung3"), "S1": inc.get("S1"), "decision": e["decision_rung"]})
pd.DataFrame(rows)

preconditions: {'P1': True, 'P2': True, 'P3': True, 'P4': True, 'P5': True, 'P6': True, 'P7': True, 'P8': True}
populations: {'chento_BTC': 208, 'chento_ETH': 184, 'squeeze_bull': 122, 'short_squeeze': 71} | I-eligible: {'chento_BTC': 164, 'chento_ETH': 154, 'squeeze_bull': 111, 'short_squeeze': 55}
squeeze_bull vs stage 1: {'stage1_trades': 122, 'trades_now': 122, 'shared': 121, 'only_in_stage1': ['BTC:1788530400'], 'only_now': ['BTC:1788526800']}
walker identity: {'i0': 0, 'x': 0, 'kind': 0, 'x_notime': 0, 'kind_notime': 0, 'level_valid': 0, 'exit_price': 0, 'exit_price_notime': 0, 'level': 0} compared 584
DVOL: {'BTC': ('2022-09-07', '2026-09-17', 1472, 0), 'ETH': ('2022-09-07', '2026-09-17', 1472, 0)}


,subpop,kind,eligible,events,median h,rung1,rung2,rung3,S1,decision
0,chento_BTC,I1_implied,164,80,13.308333,29,25,49,36,rung3
1,chento_BTC,I1_realised,164,91,12.933333,36,31,56,41,rung1
2,chento_BTC,I2_day,164,89,6.150000,53,47,66,59,rung1
3,chento_BTC,J_wick,208,152,7.983333,107,95,126,114,rung1
4,chento_BTC,J_nolevel,208,165,5.233333,120,116,142,122,rung1
5,chento_BTC,J_accept,208,156,7.108333,104,85,129,109,rung1
6,chento_BTC,J_shape,208,173,3.983333,140,135,155,143,rung1
7,chento_BTC,J_wick_1m,208,160,7.058333,103,98,131,107,rung1
8,chento_ETH,I1_implied,154,77,15.433333,23,22,40,31,rung3
9,chento_ETH,I1_realised,154,76,13.716667,30,27,44,33,rung1


## 2. The decision lines, recomputed from the saved events

`Δ` is the continuation value at the first event minus the placebo mean of matched control minutes (same
subpopulation, direction, ±365 days unless rung 3, same 0.25 R profit bin, fresh extreme, no prior event, status
known). Negative means the rest of the trade was worth less after the event than at matched moments. The Holm p
tests `Δ < 0`.

In [3]:
def recompute(subpop, kind):
    sub = events[(events["subpop"] == subpop) & (events["kind"] == kind)]
    sub = sub[np.isfinite(sub["placebo"])].sort_values("entry_ts", kind="mergesort")
    axis = M.day_axis(trades[trades["subpop"] == subpop]["entry_day"])
    idx = M.block_indices(len(axis))
    return M.summarize(sub, "delta", axis, idx)

keys = ["chento_BTC:I1_implied", "chento_BTC:J_wick", "squeeze_bull:J_wick", "chento_ETH:I1_implied",
        "chento_ETH:J_wick", "squeeze_bull:I1_implied", "chento_BTC:I1_realised", "chento_BTC:J_nolevel",
        "squeeze_bull:J_nolevel", "chento_ETH:I1_realised", "chento_ETH:J_nolevel"]
checked = []
for key in keys:
    pop, kind = key.split(":")
    t = report["tests"][key]; saved = t["lines"][t["decision_rung"]]
    if not saved.get("n"):
        checked.append({"test": key, "n": 0}); continue
    mine = recompute(pop, kind)
    assert mine["n"] == saved["n"], (key, mine["n"], saved["n"])
    assert abs(mine["mean"] - saved["mean"]) < 1e-9, key
    assert max(abs(a - b) for a, b in zip(mine["ci95"], saved["ci95"])) < 1e-9, key
    checked.append({"test": key, "rung": t["decision_rung"], "n": saved["n"], "Δ": round(saved["mean"], 3),
                    "95% low": round(saved["ci95"][0], 3), "95% high": round(saved["ci95"][1], 3),
                    "p(Δ<0)": round(saved["p_one_sided_less"], 3),
                    "first half": round(saved["first_half"], 2), "second half": round(saved["second_half"], 2),
                    "holding after": round(saved["mean_cv_at_event"], 3), "matched": round(saved["mean_placebo"], 3),
                    "Holm p": None if t.get("holm_p") is None else round(t["holm_p"], 3),
                    "label": t["classification"]})
print("recomputed from events.csv.gz and matched report.json exactly")
pd.DataFrame(checked)

recomputed from events.csv.gz and matched report.json exactly


,test,rung,n,Δ,95% low,95% high,p(Δ<0),first half,second half,holding after,matched,Holm p,label
0,chento_BTC:I1_implied,rung3,49,0.256,-0.466,1.035,0.757,0.98,-0.50,0.911,0.655,1.0,UNDETERMINED
1,chento_BTC:J_wick,rung1,107,0.371,-0.218,0.947,0.897,0.46,0.28,0.893,0.521,1.0,UNDETERMINED
2,squeeze_bull:J_wick,rung1,51,0.077,-0.201,0.396,0.700,0.15,0.00,0.207,0.130,1.0,UNDETERMINED
3,chento_ETH:I1_implied,rung3,40,-0.921,-1.990,0.309,0.048,-0.74,-1.10,0.326,1.247,NaN,DESCRIPTIVE
4,chento_ETH:J_wick,rung1,77,0.187,-0.636,1.101,0.664,-0.45,0.85,0.770,0.582,NaN,DESCRIPTIVE
5,squeeze_bull:I1_implied,rung1,7,-0.080,-0.135,-0.016,0.011,-0.11,-0.04,0.323,0.403,NaN,DESCRIPTIVE
6,chento_BTC:I1_realised,rung3,56,0.067,-0.704,0.846,0.595,0.98,-0.85,0.801,0.735,NaN,DESCRIPTIVE
7,chento_BTC:J_nolevel,rung1,120,-0.082,-0.568,0.438,0.393,-0.18,0.01,0.668,0.751,NaN,DESCRIPTIVE
8,squeeze_bull:J_nolevel,rung1,72,0.006,-0.187,0.215,0.536,-0.04,0.05,0.268,0.262,NaN,DESCRIPTIVE
9,chento_ETH:I1_realised,rung3,44,0.097,-1.000,1.165,0.565,0.76,-0.56,0.922,0.825,NaN,DESCRIPTIVE


## 3. Every placebo, the controls and the shared-pool contrast

If a result were an artefact of one matching rule it would move when the rule changes. The lines: rungs 1–3
(fresh / strict extreme / all years), S1–S2 (no freshness match), S3 (no bin), S4 (no-time-exit walk), R-vol and
R-session (the robustness constraints), and the shared pool used for the event-versus-control contrast.

In [4]:
def lines_of(key):
    t = report["tests"][key]; rung = t["decision_rung"]
    order = ["rung1", "rung2", "rung3", "S1", "S2", "S3", "S4", f"rvol_{rung}", f"rsession_{rung}", f"shared_{rung}"]
    return {n.replace(f"_{rung}", ""): (f"{t['lines'][n]['mean']:+.3f} (n {t['lines'][n]['n']})"
                                        if t["lines"].get(n, {}).get("n") else None) for n in order if n in t["lines"]}
pd.DataFrame({k: lines_of(k) for k in ["chento_BTC:I1_implied", "chento_BTC:I1_realised", "chento_BTC:J_wick",
                                        "chento_BTC:J_nolevel", "squeeze_bull:J_wick", "squeeze_bull:J_nolevel",
                                        "chento_ETH:I1_implied", "chento_ETH:J_wick"]})

,chento_BTC:I1_implied,chento_BTC:I1_realised,chento_BTC:J_wick,chento_BTC:J_nolevel,squeeze_bull:J_wick,squeeze_bull:J_nolevel,chento_ETH:I1_implied,chento_ETH:J_wick
rung1,+0.217 (n 29),+0.354 (n 36),+0.371 (n 107),-0.082 (n 120),+0.077 (n 51),+0.006 (n 72),-1.026 (n 23),+0.187 (n 77)
rung2,+0.325 (n 25),+0.122 (n 31),+0.367 (n 95),-0.093 (n 116),+0.047 (n 44),+0.013 (n 69),-0.933 (n 22),+0.183 (n 62)
rung3,+0.256 (n 49),+0.067 (n 56),+0.270 (n 126),+0.091 (n 142),+0.087 (n 58),+0.015 (n 80),-0.921 (n 40),+0.145 (n 92)
S1,+0.131 (n 36),+0.367 (n 41),+0.352 (n 114),-0.049 (n 122),-0.033 (n 55),+0.008 (n 75),-1.087 (n 31),+0.332 (n 83)
S2,+0.368 (n 57),+0.234 (n 59),+0.231 (n 133),+0.008 (n 146),+0.006 (n 66),-0.013 (n 82),-0.921 (n 47),+0.227 (n 99)
S3,+0.294 (n 67),+0.257 (n 72),+0.248 (n 127),+0.093 (n 140),+0.107 (n 60),+0.121 (n 79),-0.363 (n 50),+0.212 (n 96)
S4,+0.114 (n 29),-0.414 (n 36),+0.219 (n 107),-0.086 (n 120),+0.014 (n 51),+0.024 (n 72),-1.117 (n 23),-0.201 (n 77)
rvol,+0.238 (n 48),+0.141 (n 54),+0.317 (n 97),-0.169 (n 110),+0.068 (n 50),+0.024 (n 65),-0.872 (n 39),+0.392 (n 71)
rsession,+0.550 (n 11),-0.320 (n 19),+0.470 (n 40),-0.417 (n 71),-0.061 (n 21),+0.038 (n 44),-1.069 (n 11),-0.312 (n 36)
shared,+0.330 (n 72),+0.113 (n 85),+0.078 (n 126),+0.107 (n 146),+0.066 (n 60),+0.013 (n 86),-0.540 (n 56),+0.163 (n 91)


In [5]:
diff = {k: report["tests"][k].get("shared_difference") for k in report["family"]}
pd.DataFrame({k: {"n event": v["n_a"], "n control": v["n_b"], "Δ event − Δ control": round(v["mean"], 3),
                  "95% low": round(v["ci95"][0], 3), "95% high": round(v["ci95"][1], 3)}
              for k, v in diff.items() if v and "mean" in v})

,chento_BTC:I1_implied,chento_BTC:J_wick,squeeze_bull:J_wick
n event,72.000,126.000,60.000
n control,85.000,146.000,86.000
Δ event − Δ control,0.217,-0.029,0.053
95% low,0.002,-0.282,-0.119
95% high,0.460,0.215,0.242


## 4. The reported kinds, the ETH replication and the exit-arm overlay

The overlay is the arm the brainstorm and the Paladin study phrased: exit at the close of the first event minute,
otherwise the shipped exit, paired per trade against the shipped exit over the whole subpopulation (non-event trades
contribute zero). It decides nothing; a positive overlay with a null `Δ` is the drift of the matched placebo.

In [6]:
rep = []
for key, t in report["tests"].items():
    pop, kind = key.split(":")
    if kind in ("I1_implied", "I1_realised", "J_wick", "J_nolevel"):
        continue
    line = t["lines"].get(t["decision_rung"], {})
    if line.get("n"):
        rep.append({"test": key, "rung": t["decision_rung"], "n": line["n"], "Δ": round(line["mean"], 3),
                    "95% low": round(line["ci95"][0], 3), "95% high": round(line["ci95"][1], 3),
                    "halves": f"{line['first_half']:+.2f} / {line['second_half']:+.2f}"})
pd.DataFrame(rep)

,test,rung,n,Δ,95% low,95% high,halves
0,chento_BTC:I1_implied_only,rung3,10,0.673,-0.342,1.717,+0.07 / +1.27
1,chento_BTC:I2_day,rung3,66,-0.191,-0.855,0.497,+0.24 / -0.62
2,chento_BTC:I2_day_realised,rung3,79,0.047,-0.615,0.726,+0.70 / -0.62
3,chento_BTC:J_accept,rung1,104,0.093,-0.555,0.732,+0.18 / +0.01
4,chento_BTC:J_shape,rung1,140,0.147,-0.343,0.658,+0.28 / +0.01
5,chento_BTC:J_wick_1m,rung1,103,-0.068,-0.663,0.529,-0.28 / +0.14
6,chento_BTC:J_nolevel_1m,rung1,132,0.155,-0.427,0.748,-0.15 / +0.46
7,chento_ETH:I1_implied_only,rung3,14,0.664,-1.274,2.164,-0.14 / +1.47
8,chento_ETH:I2_day,rung3,58,-0.616,-1.460,0.411,-0.62 / -0.62
9,chento_ETH:I2_day_realised,rung3,66,-0.317,-1.203,0.617,-0.02 / -0.61


In [7]:
print("ETH non-overlap subsets:", json.dumps(report["eth_non_overlap"], indent=1))
ov = report["overlay"]
pd.DataFrame({k: {"n": v["n"], "events": v["n_event"], "mean Δ R (overlay − shipped)": round(v["mean_diff_R"], 3),
                  "95% low": round(v["ci95"][0], 3), "95% high": round(v["ci95"][1], 3),
                  "shipped R": round(v["mean_shipped_R"], 3), "overlay R": round(v["mean_arm_R"], 3),
                  "event trades that stopped out": v["event_trades_shipped_stop"],
                  "event trades that hit target": v["event_trades_shipped_target"]}
              for k, v in ov.items() if v.get("n")}).T

ETH non-overlap subsets: {
 "I1_implied": {
  "eth_event_trades": 77,
  "overlapping": 33,
  "subset": 44,
  "subset_included": 18,
  "subset_mean_delta": -1.1393427616723302
 },
 "J_wick": {
  "eth_event_trades": 118,
  "overlapping": 52,
  "subset": 66,
  "subset_included": 40,
  "subset_mean_delta": -0.4346752259410039
 }
}


,n,events,mean Δ R (overlay − shipped),95% low,95% high,shipped R,overlay R,event trades that stopped out,event trades that hit target
chento_BTC:I1_implied,208.0,80.0,-0.277,-0.526,-0.045,0.788,0.511,8.0,16.0
chento_BTC:J_wick,208.0,152.0,-0.549,-0.910,-0.203,0.788,0.239,44.0,23.0
chento_ETH:I1_implied,184.0,77.0,-0.175,-0.467,0.106,0.584,0.409,15.0,11.0
chento_ETH:J_wick,184.0,118.0,-0.424,-0.804,-0.062,0.584,0.160,36.0,15.0
squeeze_bull:I1_implied,122.0,15.0,-0.010,-0.045,0.026,0.305,0.295,0.0,13.0
squeeze_bull:J_wick,122.0,70.0,-0.088,-0.203,0.030,0.305,0.217,8.0,36.0
short_squeeze:I1_implied,71.0,0.0,0.000,0.000,0.000,0.346,0.346,0.0,0.0
short_squeeze:J_wick,71.0,14.0,-0.031,-0.145,0.084,0.346,0.316,2.0,6.0


## 5. Base rates, the exit-minute channel and covariate balance

The channel: a trade whose first pattern minute is its exit minute has no event and serves as a control. For I1 on
squeeze_bull the target (+3 %) sits about one implied day above the flush, so the crossing and the target fill
coincide on 7 of 15 event candidates; that is why squeeze_bull's I1 is descriptive.

In [8]:
print("exit-minute channel:", json.dumps({n: {k: v for k, v in c.items() if k in ("I1_implied", "J_wick")}
                                          for n, c in pre["P8"]["exit_minute_channel"].items()}, indent=1))
pd.DataFrame({n: per["_base_rates"] for n, per in pre["P8"]["per_subpop"].items()}).T

exit-minute channel: {
 "chento_BTC": {
  "I1_implied": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  },
  "J_wick": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  }
 },
 "chento_ETH": {
  "I1_implied": {
   "target": 1,
   "stop": 0,
   "time": 0,
   "other": 0
  },
  "J_wick": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  }
 },
 "squeeze_bull": {
  "I1_implied": {
   "target": 7,
   "stop": 0,
   "time": 0,
   "other": 0
  },
  "J_wick": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  }
 },
 "short_squeeze": {
  "I1_implied": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  },
  "J_wick": {
   "target": 0,
   "stop": 0,
   "time": 0,
   "other": 0
  }
 }
}


,share_eligible_trades_crossing_IMP,share_eligible_trades_crossing_RV,armed_bars,armed_bars_with_rejection_shape,share_armed_with_shape,share_shape_at_level,mean_candidate_levels_per_trade
chento_BTC,0.487805,0.554878,20166.0,5387.0,0.267133,0.291628,5.778846
chento_ETH,0.500000,0.493506,17534.0,4846.0,0.276377,0.170863,4.858696
squeeze_bull,0.135135,0.333333,5758.0,1507.0,0.261723,0.240212,4.959016
short_squeeze,0.000000,0.000000,295.0,96.0,0.325424,0.322917,4.295775


In [9]:
bal = {}
for name, per in pre["P8"]["per_subpop"].items():
    for kind in ("I1_implied", "J_wick"):
        b = per[kind].get("balance")
        if b and b.get("n") and "event" in b:
            bal[f"{name}:{kind}"] = {"n": b["n"], "VR event": round(b["event"]["vr_median"], 2),
                                     "VR controls": round(b["controls"]["vr_median"], 2),
                                     "mark event": round(b["event"]["mark_median"], 2),
                                     "mark controls": round(b["controls"]["mark_median"], 2),
                                     "ord event q": b["event"]["ord_q"], "ord controls q": b["controls"]["ord_q"],
                                     "weekend event": round(b["event"]["weekend_share"], 2),
                                     "weekend controls": round(b["controls"]["weekend_share"], 2)}
pd.DataFrame(bal).T

,n,VR event,VR controls,mark event,mark controls,ord event q,ord controls q,weekend event,weekend controls
chento_BTC:I1_implied,80,2.48,2.09,1.25,0.83,"[20.0, 27.0, 39.0]","[19.0, 25.0, 34.0]",0.39,0.31
chento_BTC:J_wick,152,1.99,0.91,0.49,0.3,"[15.0, 20.0, 26.0]","[11.0, 15.0, 21.0]",0.36,0.4
chento_ETH:I1_implied,77,2.61,1.98,1.23,0.82,"[19.0, 26.0, 32.0]","[16.0, 22.0, 28.0]",0.26,0.37
chento_ETH:J_wick,118,1.95,0.98,0.55,0.24,"[14.0, 21.0, 29.0]","[8.0, 12.0, 18.0]",0.26,0.25
squeeze_bull:J_wick,70,1.24,0.77,0.58,0.42,"[17.0, 24.5, 34.5]","[13.0, 18.0, 26.0]",0.29,0.34


## 6. Verdict

Section 8's rule, applied by `rangewick_run.stage_verdict`: a kind is promoted only with an INFORMATIVE family test
and an agreeing chento-ETH line. The reading is in [findings_range_wick.md](findings_range_wick.md).

In [10]:
print(json.dumps(verdict, indent=1))

{
 "created_utc": "2026-09-18T23:27:14+00:00",
 "promoted": [],
 "per_kind": {
  "I1_implied": {
   "informative_family_tests": [],
   "replication": "not evaluated"
  },
  "J_wick": {
   "informative_family_tests": [],
   "replication": "not evaluated"
  }
 },
 "classifications": {
  "chento_BTC:I1_implied": "UNDETERMINED",
  "chento_BTC:J_wick": "UNDETERMINED",
  "squeeze_bull:J_wick": "UNDETERMINED"
 },
 "verdict": "NONE PROMOTED",
 "permits": "nothing in production; a promotion permits only a separate stage 2 pre-registration"
}
